# 第 5 章 · 多智能体与声明式工作流

> **你已会**：Agent、工具、显式数据契约，以及第 3 章的 State 最小机制。
>
> **读完后你能**：用当前 Workflow 写出顺序、条件和汇合；区分程序路由、模型转移与 AgentTool 调用；说明何时值得拆 Agent。
>
> **边界**：本章主线不调用模型。动态节点在第 8 章，恢复在第 9 章，父子回调实验在第 7 章。

主线约 45 分钟：先运行三个离线小实验，再讨论分工。旧 Sequential/Parallel/Loop 只在迁移深读中，不是新项目的默认起点。

## 1. 先不拆 Agent：把业务步骤看清楚

任务是处理订单 A100 的退款查询。先读取请求，再查订单和金额，最后给出是否需要继续审核的结论；本章不执行付款。

**先预测**：步骤顺序、路由条件和分支数量都已知时，是否需要再调用模型决定下一步？

下面的实现使用真实 ADK Runner 与事件，没有网络或模型噪声。代码在 `graph_example/workflow.py`；每次请求都创建独立 Session。

In [1]:
import sys
from pathlib import Path

lesson_root = Path.cwd()
if not (lesson_root / "graph_example").exists():
    lesson_root /= "google-adk"
if str(lesson_root) not in sys.path:
    sys.path.insert(0, str(lesson_root))

from graph_example.workflow import (
    build_sequence, build_branch, build_fanout, run_graph, verify,
)
request = {"order_id": "A100", "amount": 5000}  # 金额单位：分

## 2. 顺序：数据从上一节点流到下一节点

`Workflow(edges=[("START", read_request, summarize)])` 就是声明路径。入口把 Runner 的 Content 转成字典；后一步通过 `node_input` 接收前一步输出，不用把临时数据写进 State。

代码里的节点用 `@node` 装饰：它把普通函数登记为框架节点。函数的 `node_input` 参数接收上一节点的输出，`ctx: Context` 由框架注入（用于设置路由等运行期信息）。第 8 章会展开 `ctx.run_node()`。

```python
from google.adk import Context
from google.adk.workflow import node

@node
def choose_route(ctx: Context, node_input: dict) -> dict:
    ctx.route = "manual" if node_input["amount"] > 10000 else "automatic"
    return node_input
```

**预测**：若不设置 output_key，下一节点还能得到上一步的字典吗？

In [2]:
flow = build_sequence()
print("声明的边：", [(edge.from_node.name, edge.to_node.name) for edge in flow.graph.edges])
result, events = await run_graph(flow, request)
assert result["summary"] == "订单 A100 申请退款"
assert result["amount"] == 5000
for event in events:
    if event.node_info and event.output is not None:
        print(event.node_info.path, "→", event.output)

声明的边： [('__START__', 'read_request'), ('read_request', 'summarize')]
sequence@1/read_request@1 → {'order_id': 'A100', 'amount': 5000}
sequence@1/summarize@1 → {'order_id': 'A100', 'amount': 5000, 'summary': '订单 A100 申请退款'}


**证据**：summarize 得到 amount 并生成 summary。显式输出可以沿图传递，State 仍用于工具或回调共享的数据。

## 3. 条件：只执行被选中的分支

在 choose_route 中写 `ctx.route = "manual" if amount > 10000 else "automatic"`，在 edges 中把字符串映射到节点。返回值负责传数据，route 负责选路径，两件事分开。

**预测**：5000 分与 20000 分分别会执行哪个节点？小额请求会不会也执行人工审核节点？

In [3]:
small, small_events = await run_graph(build_branch(), request)
large, large_events = await run_graph(build_branch(), {**request, "amount": 20000})
assert small["status"] == "eligible_for_next_check"
assert large["status"] == "needs_review"
small_paths = [event.node_info.path for event in small_events if event.node_info]
assert not any("manual_review@" in path for path in small_paths)
print({"小额": small["status"], "大额": large["status"]})

{'小额': 'eligible_for_next_check', '大额': 'needs_review'}


**证据**：不仅返回值不同，小额请求的执行事件中也没有 manual_review。业务状态是受控规则，不是一次模型判断。

## 4. 并行与汇合：两个结果都到齐才下结论

`(check_order, check_amount)` 声明扇出；`JoinNode(name="both_checks")` 等待两路结果。汇合输入按节点名组织，不能用完成顺序猜结果是谁的。

`max_concurrency=2` 限制图调度节点。第 8 章的动态调用有不同的并发控制位置。

**预测**：若订单不存在、金额仍在限额内，汇合结果应该成功吗？

In [4]:
valid, _ = await run_graph(build_fanout(), request)
missing, _ = await run_graph(build_fanout(), {**request, "order_id": "missing"})
assert valid["passed"] is True and missing["passed"] is False
assert valid["sources"] == ["check_amount", "check_order"]
print({"正常订单": valid, "缺失订单": missing})
await verify()

{'正常订单': {'order_id': 'A100', 'amount': 5000, 'passed': True, 'sources': ['check_amount', 'check_order']}, '缺失订单': {'order_id': 'missing', 'amount': 5000, 'passed': False, 'sources': ['check_amount', 'check_order']}}
✅ 声明式顺序、条件两条路径、并行汇合与反例均通过（0 次模型调用）


**反例**：金额通过不代表订单存在。JoinNode 保证等待关系；业务正确性来自 combine_checks 的明确规则。此实验未测性能，因此不能由“有两个分支”推断一定更快。

## 5. 什么时候再引入多个 Agent

| 问题 | 优先考虑 | 为什么 |
|---|---|---|
| 步骤和规则已知 | 声明式 Workflow | 路径可以直接检查，少一次模型路由 |
| 输入决定任务数量或复杂控制流 | 第 8 章动态调用 | 用 Python 表达运行时控制 |
| 需要借用另一个 Agent 的能力，完成后继续组织回答 | `mode="single_turn"` 子 Agent（框架自动暴露为工具；`AgentTool` 亦可） | 调用后返回主 Agent |
| 需要把对话交给一个专职 Agent 接管 | sub_agents / transfer_to_agent | 改变后续处理者 |

“谁表达路径”和“谁作决策”是两个维度。Agent 可以作为 Workflow 中一个完成语言任务的节点；图本身也可以只含普通 Python 节点。

不要因为 prompt 变长就自动拆 Agent。先检查能否用一个 Agent 加清楚的工具契约解决；确需独立职责、上下文或并行工作时再拆。拆分还会增加调用成本和交接错误。

### 5.1 可选：用真实模型观察转移

第 4 章已有 AgentTool。下面的旧版路由示例仍可用于观察 `transfer_to_agent`，但它不是前三个图实验的先修；需要 DeepSeek 密钥。

<details>
<summary>深入：真实模型委派示例（复制到独立单元运行）</summary>

```python
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "可选委派实验需要密钥"

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER = "adk_ch04", "student"
DS = lambda: LiteLlm(model="deepseek/deepseek-chat")  # 每个 Agent 独立模型实例

async def run_team(root_agent, query, session_id="demo", show_state_keys=None):
    """运行一个（可能是多智能体的）Agent 系统，打印接力过程，返回 session。"""
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=session_id)
    runner = Runner(agent=root_agent, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 {query}\n" + "═" * 60)
    async for ev in runner.run_async(user_id=USER, session_id=session_id, new_message=msg):
        if not (ev.content and ev.content.parts):
            continue
        for p in ev.content.parts:
            if p.function_call:
                print(f"  ⚡ [{ev.author}] 调用 {p.function_call.name}({dict(p.function_call.args)})")
            elif p.function_response:
                print(f"  📦 [{ev.author}] {p.function_response.name} 返回（略）")
            elif p.text:
                tag = "🏁 最终" if ev.is_final_response() else "🗣️ 中间"
                print(f"  {tag} [{ev.author}] {p.text[:120]}")
    session = await ss.get_session(app_name=APP, user_id=USER, session_id=session_id)
    if show_state_keys:
        print("═" * 60, "\n🗂️ Session 状态快照：")
        for k in show_state_keys:
            v = session.state.get(k, "（无）")
            print(f"  {k} = {str(v)[:100]}")
    return session

print("✅ 环境就绪")

academic = Agent(
    name="academic_affairs", model=DS(),
    instruction="你是教务处专家，用中文简洁回答课表、考试、学分相关问题。",
    description="负责教务问题：课程表、考试安排、学分与成绩政策。",  # ← 路由依据！
)

it_support = Agent(
    name="it_support", model=DS(),
    instruction="你是校园 IT 支持，用中文给出账号、网络、设备问题的排查步骤，分点说明。",
    description="负责 IT 问题：校园网、账号登录、邮箱与电脑故障。",   # ← 路由依据！
)

front_desk = Agent(
    name="front_desk", model=DS(),
    instruction="你是服务台前台，自己不回答问题，必须根据问题类型把对话转交给对应专家。",
    description="校园服务台总机",
    sub_agents=[academic, it_support],   # ← 声明可委派的对象
)

await run_team(front_desk, "你好，我连不上校园网了，而且想问问下周的高数考试安排。", session_id="route")

```

观察转移事件及下一轮由谁处理；不要把一次路由成功当成路由准确率。练习需包含含糊和复合请求。

</details>

## 6. 从旧工作流 Agent 迁移

| 旧写法 | 当前主线 | 数据变化 |
|---|---|---|
| SequentialAgent | edges 中声明链 | output_key 可以改成节点输出 |
| ParallelAgent | 扇出 + JoinNode | 按来源节点汇合返回值 |
| LoopAgent | 明确的条件/循环，或第 8 章有界循环 | 退出成功与预算耗尽分别表达 |

以下旧 API 在锁定版本仍可运行，但已标记 deprecated。保留它们是为了阅读已有工程；不能因此断言新 Workflow 与所有旧 sub_agents 嵌套用法等价。

<details>
<summary>迁移深读：旧 Sequential / Parallel / Loop 示例</summary>

---

## 迁移 1 · SequentialAgent：串行流水线

**场景**：内容生产流水线——`大纲 → 初稿 → 润色`，每一步由专职 Agent 完成，上一步的产出通过 `output_key` 存入状态，下一步用 `{key}` 插值读取。

```mermaid
flowchart LR
    U["主题输入"] --> A["1️⃣ 大纲师<br/>output_key=outline"]
    A -->|"state['outline']"| B["2️⃣ 撰稿人<br/>读{outline}<br/>output_key=draft"]
    B -->|"state['draft']"| C["3️⃣ 润色师<br/>读{draft}<br/>output_key=final"]
    C --> O["成品文案"]
    style A fill:#e8f0fe,stroke:#4285f4
    style B fill:#e6f4ea,stroke:#34a853
    style C fill:#fef7e0,stroke:#fbbc04
```

这就是第 3 章 `output_key` + `{state插值}` 的实战组合：


```python
from google.adk.agents import SequentialAgent

outliner = Agent(
    name="outliner", model=DS(),
    instruction="你是写作大纲师。根据用户主题列出 3 个要点的大纲，每点一句话。只输出大纲。",
    description="生成文章大纲",
    output_key="outline",
)

writer = Agent(
    name="writer", model=DS(),
    instruction="你是撰稿人。根据下面的大纲写一段 150 字左右的短文：\n{outline}\n只输出正文。",
    description="根据大纲写初稿",
    output_key="draft",
)

polisher = Agent(
    name="polisher", model=DS(),
    instruction="你是润色师。把下面的初稿改得更生动（保持字数，加入一个比喻）：\n{draft}\n只输出润色后的成稿。",
    description="润色初稿成成品",
    output_key="final",
)

pipeline = SequentialAgent(
    name="writing_pipeline",
    sub_agents=[outliner, writer, polisher],  # 按列表顺序依次执行
)

session = await run_team(pipeline, "主题：程序员为什么需要学习写作", session_id="seq",
                         show_state_keys=["outline", "final"])

```

> 🔍 观察输出中每个事件的作者：`outliner → writer → polisher` 严格按序接力，状态在三人之间无缝传递。**整个过程没有任何 LLM 路由决策**——这就是"确定性编排"的可靠感。

---

## 迁移 2 · ParallelAgent：并行扇出

**场景**：商业分析——同一命题，**技术、市场、风险**三个视角同时开分析会，最后由汇总员合并结论。

```mermaid
flowchart TD
    U["命题输入"] --> P{"ParallelAgent<br/>同时启动"}
    P --> T["🔬 技术分析师<br/>output_key=tech"]
    P --> M["📈 市场分析师<br/>output_key=market"]
    P --> K["⚠️ 风险分析师<br/>output_key=risk"]
    T --> S["🧾 汇总员<br/>读三个 key 合并"]
    M --> S
    K --> S
    style P fill:#fef7e0,stroke:#fbbc04,stroke-width:2px
```

> ⚠️ **关键规则**：并行分支共享同一份 Session 状态，所以每个分支必须写**不同的** `output_key`，否则会互相覆盖。

并行 + 串行可以**自由嵌套**——这正是 ADK 工作流的组合美学：


```python
from google.adk.agents import ParallelAgent

def make_analyst(name, angle, key):
    return Agent(
        name=name, model=DS(),
        instruction=f"你是{angle}分析师。只从{angle}视角分析用户给的命题，80 字以内，直入主题。",
        description=f"{angle}视角分析",
        output_key=key,
    )

brainstorm = ParallelAgent(
    name="brainstorm",
    sub_agents=[
        make_analyst("tech_analyst", "技术可行性", "tech"),
        make_analyst("market_analyst", "市场前景", "market"),
        make_analyst("risk_analyst", "潜在风险", "risk"),
    ],
)

merger = Agent(
    name="merger", model=DS(),
    instruction="""你是首席顾问。综合三位分析师的意见给出一段总结与一句行动建议：
【技术】{tech}
【市场】{market}
【风险】{risk}""",
    description="汇总多视角分析",
    output_key="report",
)

# Sequential 套 Parallel：先并行扇出，再串行汇总
think_tank = SequentialAgent(name="think_tank", sub_agents=[brainstorm, merger])

session = await run_team(think_tank, "命题：在高校推广 AI 编程助教", session_id="par",
                         show_state_keys=["report"])

```

---

## 迁移 3 · LoopAgent：迭代打磨循环

**场景**：文案打磨——`写手` 与 `毒舌评审` 反复过招，直到评审满意为止。循环的**退出机制**有两种：

1. `max_iterations`：兜底上限，防止死循环；
2. **主动退出**：某个 Agent 调用一个设置 `tool_context.actions.escalate = True` 的工具，循环立即终止。

```mermaid
flowchart LR
    W["✍️ 写手<br/>写/改文案"] --> C["🧐 评审<br/>打分"]
    C -->|"不满意：给意见"| W
    C -->|"满意：调用 exit_loop"| E(["✅ 退出循环"])
    L["LoopAgent<br/>max_iterations=4 兜底"] -.包裹.-> W
    style L fill:#fce8e6,stroke:#ea4335,stroke-width:2px
```


```python
from google.adk.agents import LoopAgent
from google.adk.tools import ToolContext

def exit_loop(tool_context: ToolContext) -> dict:
    """当文案质量已经达标时调用此工具，立即结束迭代循环。"""
    tool_context.actions.escalate = True   # 向 LoopAgent 发出"收工"信号
    return {"status": "评审通过，循环结束"}

copywriter = Agent(
    name="copywriter", model=DS(),
    instruction="你是广告写手。根据主题写一句 slogan；如果会话中已有评审意见，必须针对意见修改后再提交。只输出 slogan 本身。",
    description="撰写/修改 slogan",
    output_key="slogan",
)

critic = Agent(
    name="critic", model=DS(),
    instruction="""你是毒舌评审。审查当前 slogan：{slogan}
- 如果朗朗上口且有记忆点 → 直接调用 exit_loop 工具，不要输出多余文字；
- 否则 → 用一句话指出问题（存入你的回复），供写手下一轮修改。""",
    description="评审 slogan 并决定是否收工",
    tools=[exit_loop],
)

polish_loop = LoopAgent(
    name="polish_loop",
    sub_agents=[copywriter, critic],  # 每一轮：先写后评
    max_iterations=4,                 # 兜底：最多 4 轮
)

session = await run_team(polish_loop, "主题：无糖气泡水，主打『0 负担的快乐』", session_id="loop",
                         show_state_keys=["slogan"])

```

这些示例需要各自的运行辅助函数和模型配置，不向主线提供变量；新项目优先使用前面的 Workflow 实验。
</details>

## 用五句话复述

1. Workflow 既能编排 Agent，也能编排普通函数。
2. edges 决定图调度路径，node_input 接收上一步数据。
3. route 选分支，返回值传数据。
4. JoinNode 等待分支并按来源汇合，不替业务做正确性判断。
5. `mode="single_turn"` 子 Agent / AgentTool 是调用-返回；转移用于交接对话，二者不是同一控制关系。

## 练习：从照着跑到独立改

1. **补全**：为请求加入 currency，保证每一步保留它，断言结果。
2. **排错**：让金额检查错误地总是通过，写一个测试使它失败，再修复。
3. **迁移**：把“订单+金额”改成“库存+地址”，不用复制旧 State 键传值。
4. **设计**：说明哪一步确实需要模型，写出输入/输出契约及一次失败应如何处理。
5. **旧工程阅读**：可读 practice/c5_4.py 的旧嵌套客服示例，列出迁移到当前 Workflow 时哪些接口不能直接替换。

验收入口：从 google-adk 目录运行 `python -m graph_example.workflow`。通过只证明编排路径和规则，不证明退款业务或模型质量。